# Ejercicio 3: Consultas directas sobre archivos Parquet

Cada consulta vive en `sql/ejercicio3/` y se ejecuta con DuckDB directamente sobre `data/raw/<tipo>/2026/*.parquet`, sin importar los datos a una tabla.

La documentacion de cada consulta (objetivo, fuentes, resultado y decisiones) esta en `docs/ejercicio3.md`.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

# Las rutas de los .sql son relativas a la raiz del proyecto
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
DIR_SQL = Path("sql/ejercicio3")


def ejecutar(nombre: str) -> pd.DataFrame:
    consulta = (DIR_SQL / f"{nombre}.sql").read_text()
    print(consulta)
    return duckdb.sql(consulta).df()

## 3.1 Cantidad de archivos

In [2]:
ejecutar("01_conteo_archivos")

-- 3.1 Cantidad de archivos Parquet disponibles por tipo de taxi
SELECT regexp_extract(file, 'raw/(\w+)/', 1) AS tipo,
       count(*) AS archivos,
       min(regexp_extract(file, '(\d{4}-\d{2})\.parquet', 1)) AS primer_mes,
       max(regexp_extract(file, '(\d{4}-\d{2})\.parquet', 1)) AS ultimo_mes
FROM glob('data/raw/*/2026/*.parquet')
GROUP BY ALL
ORDER BY tipo;



,tipo,archivos,primer_mes,ultimo_mes
0,green,8,2026-01,2026-08
1,yellow,8,2026-01,2026-08


## 3.2 Cantidad de registros

In [3]:
ejecutar("02_registros_metadata")

-- 3.2 Registros por archivo leidos del footer Parquet (sin escanear los datos)
SELECT regexp_extract(file_name, 'raw/(\w+)/', 1) AS tipo,
       regexp_extract(file_name, '(\d{4}-\d{2})\.parquet', 1) AS mes,
       num_rows AS registros,
       num_row_groups AS row_groups
FROM parquet_file_metadata('data/raw/*/2026/*.parquet')
ORDER BY tipo, mes;



,tipo,mes,registros,row_groups
0,green,2026-01,40272,1
1,green,2026-02,37373,1
2,green,2026-03,44208,1
3,green,2026-04,44238,1
4,green,2026-05,44921,1
5,green,2026-06,44163,1
6,green,2026-07,41252,1
7,green,2026-08,40687,1
8,yellow,2026-01,3724889,4
9,yellow,2026-02,3399866,4


In [4]:
ejecutar("03_registros_conteo")

-- 3.2 Registros por tipo contando filas (contraste con la metadata)
SELECT regexp_extract(filename, 'raw/(\w+)/', 1) AS tipo,
       count(*) AS registros
FROM read_parquet('data/raw/*/2026/*.parquet', filename = true, union_by_name = true)
GROUP BY ALL
UNION ALL
SELECT 'TOTAL', count(*)
FROM read_parquet('data/raw/*/2026/*.parquet', union_by_name = true)
ORDER BY tipo;



,tipo,registros
0,TOTAL,30040469
1,green,337114
2,yellow,29703355


## 3.3 / 3.4 Columnas y tipos de datos

In [5]:
ejecutar("04_columnas_yellow")

-- 3.3 / 3.4 Columnas y tipos (DuckDB) de los archivos yellow
DESCRIBE SELECT * FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true);



,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [6]:
ejecutar("05_columnas_green")

-- 3.3 / 3.4 Columnas y tipos (DuckDB) de los archivos green
DESCRIBE SELECT * FROM read_parquet('data/raw/green/2026/*.parquet', union_by_name = true);



,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [7]:
ejecutar("06_comparacion_columnas")

-- 3.3 Columnas presentes en cada tipo de taxi y en cuantos archivos aparecen
SELECT name AS columna,
       count(DISTINCT file_name) FILTER (WHERE file_name LIKE '%/yellow/%') AS archivos_yellow,
       count(DISTINCT file_name) FILTER (WHERE file_name LIKE '%/green/%') AS archivos_green
FROM parquet_schema('data/raw/*/2026/*.parquet')
WHERE name <> 'schema'
GROUP BY name
ORDER BY archivos_yellow = 0, archivos_green = 0, columna;



,columna,archivos_yellow,archivos_green
0,DOLocationID,8,8
1,PULocationID,8,8
2,RatecodeID,8,8
3,VendorID,8,8
4,cbd_congestion_fee,8,8
5,congestion_surcharge,8,8
6,extra,8,8
7,fare_amount,8,8
8,improvement_surcharge,8,8
9,mta_tax,8,8


In [8]:
ejecutar("07_tipos_por_archivo")

-- 3.4 Tipos fisicos Parquet: columnas cuyo tipo o presencia cambia entre archivos
WITH esquema AS (
    SELECT regexp_extract(file_name, 'raw/(\w+)/', 1) AS tipo,
           regexp_extract(file_name, '(\d{4}-\d{2})\.parquet', 1) AS mes,
           name AS columna, type AS tipo_fisico, logical_type
    FROM parquet_schema('data/raw/*/2026/*.parquet')
    WHERE name <> 'schema'
)
SELECT tipo, columna,
       string_agg(DISTINCT tipo_fisico, ', ') AS tipos_fisicos,
       count(*) AS archivos_con_columna,
       string_agg(mes, ', ' ORDER BY mes) AS meses
FROM esquema
GROUP BY tipo, columna
HAVING count(DISTINCT tipo_fisico) > 1 OR count(*) < 8
ORDER BY tipo, columna;



,tipo,columna,tipos_fisicos,archivos_con_columna,meses
0,green,request_source,BYTE_ARRAY,3,"2026-06, 2026-07, 2026-08"
1,yellow,request_source,BYTE_ARRAY,3,"2026-06, 2026-07, 2026-08"


## 3.5 Muestra de registros

In [9]:
ejecutar("08_muestra_yellow")

-- 3.5 Muestra aleatoria reproducible de registros yellow
SELECT * FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true)
USING SAMPLE reservoir(10 ROWS) REPEATABLE (42);



,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-01-01 04:31:37,2026-01-01 04:36:45,3,1.01,1,N,246,48,1,7.2,1.00,0.5,2.59,0.00,1.0,15.54,2.5,0.00,0.75,None
1,2,2026-01-01 07:47:21,2026-01-01 07:49:32,1,0.39,1,N,141,141,1,4.4,0.00,0.5,1.00,0.00,1.0,9.40,2.5,0.00,0.00,None
2,2,2026-01-01 10:21:39,2026-01-01 10:24:35,1,0.72,1,N,50,246,1,5.8,0.00,0.5,2.64,0.00,1.0,13.19,2.5,0.00,0.75,None
3,1,2026-01-01 11:58:50,2026-01-01 12:07:08,1,1.70,1,N,239,161,1,10.0,3.25,0.5,2.95,0.00,1.0,17.70,2.5,0.00,0.75,None
4,2,2026-01-01 12:39:27,2026-01-01 12:52:43,1,4.79,1,N,140,232,1,21.2,0.00,0.5,5.19,0.00,1.0,31.14,2.5,0.00,0.75,None
5,2,2026-01-01 13:42:32,2026-01-01 13:54:47,1,1.37,1,N,162,68,1,12.1,0.00,0.5,3.37,0.00,1.0,20.22,2.5,0.00,0.75,None
6,2,2026-01-01 13:28:55,2026-01-01 13:37:15,3,1.43,1,N,100,164,1,9.3,0.00,0.5,2.81,0.00,1.0,16.86,2.5,0.00,0.75,None
7,2,2026-01-01 14:04:13,2026-01-01 14:27:47,4,10.19,1,N,100,138,1,42.2,5.00,0.5,2.00,6.94,1.0,60.89,2.5,0.00,0.75,None
8,1,2026-01-01 15:35:24,2026-01-01 16:07:48,1,22.00,1,N,132,60,1,80.7,0.00,0.5,22.30,6.94,1.0,111.44,0.0,0.00,0.00,None
9,2,2026-01-01 15:40:44,2026-01-01 16:26:18,1,19.12,2,N,132,79,1,70.0,0.00,0.5,5.00,0.00,1.0,81.50,2.5,1.75,0.75,None


In [10]:
ejecutar("09_muestra_green")

-- 3.5 Muestra aleatoria reproducible de registros green
SELECT * FROM read_parquet('data/raw/green/2026/*.parquet', union_by_name = true)
USING SAMPLE reservoir(10 ROWS) REPEATABLE (42);



,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-01-03 10:31:59,2026-01-03 10:35:48,N,1,75,263,1,0.63,5.80,0.0,0.5,0.00,0.0,NaN,1.0,10.05,2,1,2.75,0.0,None
1,2,2026-01-16 19:55:39,2026-01-16 20:12:48,N,1,97,61,1,3.06,18.40,1.0,0.5,0.00,0.0,NaN,1.0,20.90,2,1,0.00,0.0,None
2,2,2026-01-19 13:09:08,2026-01-19 13:29:44,N,1,65,225,1,2.98,19.80,0.0,0.5,5.00,0.0,NaN,1.0,26.30,1,1,0.00,0.0,None
3,2,2026-01-21 17:01:53,2026-01-21 17:10:44,N,1,43,263,2,0.65,9.30,2.5,0.5,2.66,0.0,NaN,1.0,15.96,1,1,0.00,0.0,None
4,2,2026-01-23 16:43:58,2026-01-23 16:57:21,N,1,223,138,1,3.53,17.70,7.5,0.5,5.34,0.0,NaN,1.0,32.04,1,1,0.00,0.0,None
5,2,2026-01-27 13:42:53,2026-01-27 14:03:40,N,1,74,151,1,2.62,19.10,0.0,0.5,5.15,0.0,NaN,1.0,25.75,1,1,0.00,0.0,None
6,1,2026-01-30 07:39:44,2026-01-30 07:45:09,N,1,74,41,0,0.60,6.50,0.0,1.5,0.00,0.0,NaN,1.0,8.00,2,1,0.00,0.0,None
7,2,2026-01-30 23:18:59,2026-01-30 23:30:41,N,1,97,66,1,1.37,11.40,1.0,0.5,2.08,0.0,NaN,1.0,15.98,1,1,0.00,0.0,None
8,2,2026-01-06 16:11:00,2026-01-06 16:35:00,NaN,<NA>,159,151,<NA>,3.16,28.59,0.0,0.5,0.00,0.0,NaN,1.0,30.09,<NA>,<NA>,NaN,0.0,None
9,6,2026-01-22 15:42:50,2026-01-22 16:03:37,NaN,<NA>,130,218,<NA>,3.23,3.00,0.0,0.5,0.00,0.0,NaN,0.3,17.26,<NA>,<NA>,NaN,0.0,None


## 3.6 Problemas de calidad de datos

In [11]:
ejecutar("10_nulos")

-- 3.6 Porcentaje de nulos por columna y tipo de taxi
WITH viajes AS (
    SELECT 'yellow' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           tpep_pickup_datetime AS pickup, tpep_dropoff_datetime AS dropoff, *
               EXCLUDE (tpep_pickup_datetime, tpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/yellow/2026/*.parquet', filename = true, union_by_name = true)
    UNION ALL BY NAME
    SELECT 'green' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           lpep_pickup_datetime AS pickup, lpep_dropoff_datetime AS dropoff, *
               EXCLUDE (lpep_pickup_datetime, lpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/green/2026/*.parquet', filename = true, union_by_name = true)
)
SELECT tipo, count(*) AS registros,
       round(100 * count(*) FILTER (WHERE passenger_count IS NULL) / count(*), 2) AS pct_null_passenger_count,
       round(100 * count(*) FILTER (WHERE RatecodeID IS 

,tipo,registros,pct_null_passenger_count,pct_null_ratecodeid,pct_null_store_and_fwd,pct_null_congestion,pct_null_payment_type,pct_null_trip_type,pct_null_ehail_fee,pct_null_request_source
0,green,337114,14.47,14.47,14.47,14.47,14.47,14.47,100.0,94.30
1,yellow,29703355,25.98,25.98,25.98,25.98,0.00,100.00,100.0,90.23


In [12]:
ejecutar("11_fechas_fuera_de_rango")

-- 3.6 Viajes cuya fecha de inicio no corresponde al mes del archivo
WITH viajes AS (
    SELECT 'yellow' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           tpep_pickup_datetime AS pickup, tpep_dropoff_datetime AS dropoff, *
               EXCLUDE (tpep_pickup_datetime, tpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/yellow/2026/*.parquet', filename = true, union_by_name = true)
    UNION ALL BY NAME
    SELECT 'green' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           lpep_pickup_datetime AS pickup, lpep_dropoff_datetime AS dropoff, *
               EXCLUDE (lpep_pickup_datetime, lpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/green/2026/*.parquet', filename = true, union_by_name = true)
)
SELECT tipo,
       count(*) FILTER (WHERE strftime(pickup, '%Y-%m') <> mes_archivo) AS fuera_del_mes,
       count(*) FILTER (WHERE year(pickup) <> 2026) AS fuera_de_2026,
       min(pick

,tipo,fuera_del_mes,fuera_de_2026,pickup_min,pickup_max
0,green,98,14,2008-12-31 17:35:31,2026-08-31 23:58:28
1,yellow,146,17,2001-01-01 09:23:58,2026-08-31 23:59:59


In [13]:
ejecutar("12_duraciones_anomalas")

-- 3.6 Duraciones imposibles o sospechosas
WITH viajes AS (
    SELECT 'yellow' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           tpep_pickup_datetime AS pickup, tpep_dropoff_datetime AS dropoff, *
               EXCLUDE (tpep_pickup_datetime, tpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/yellow/2026/*.parquet', filename = true, union_by_name = true)
    UNION ALL BY NAME
    SELECT 'green' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           lpep_pickup_datetime AS pickup, lpep_dropoff_datetime AS dropoff, *
               EXCLUDE (lpep_pickup_datetime, lpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/green/2026/*.parquet', filename = true, union_by_name = true)
)
SELECT tipo,
       count(*) FILTER (WHERE dropoff < pickup) AS dropoff_antes_pickup,
       count(*) FILTER (WHERE dropoff = pickup) AS duracion_cero,
       count(*) FILTER (WHERE dropoff - pickup > INTERVAL 24 H

,tipo,dropoff_antes_pickup,duracion_cero,mas_de_24h,duracion_max
0,green,5,229,4,1 days 16:56:22
1,yellow,10,371673,263,11 days 22:05:25


In [14]:
ejecutar("13_distancias_y_montos")

-- 3.6 Distancias y montos fuera de lo razonable
WITH viajes AS (
    SELECT 'yellow' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           tpep_pickup_datetime AS pickup, tpep_dropoff_datetime AS dropoff, *
               EXCLUDE (tpep_pickup_datetime, tpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/yellow/2026/*.parquet', filename = true, union_by_name = true)
    UNION ALL BY NAME
    SELECT 'green' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           lpep_pickup_datetime AS pickup, lpep_dropoff_datetime AS dropoff, *
               EXCLUDE (lpep_pickup_datetime, lpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/green/2026/*.parquet', filename = true, union_by_name = true)
)
SELECT tipo,
       count(*) FILTER (WHERE trip_distance = 0) AS distancia_cero,
       count(*) FILTER (WHERE trip_distance > 200) AS distancia_mayor_200mi,
       max(trip_distance) AS distancia_max,
      

,tipo,distancia_cero,distancia_mayor_200mi,distancia_max,tarifa_negativa,total_negativo,total_mayor_1000,total_min,total_max
0,green,12212,69,179830.92,999,1023,1,-501.5,1678.2
1,yellow,952231,704,328522.20,157364,161835,49,-2560.2,7053.5


In [15]:
ejecutar("14_codigos_invalidos")

-- 3.6 Valores fuera del diccionario de datos de la TLC
WITH viajes AS (
    SELECT 'yellow' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           tpep_pickup_datetime AS pickup, tpep_dropoff_datetime AS dropoff, *
               EXCLUDE (tpep_pickup_datetime, tpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/yellow/2026/*.parquet', filename = true, union_by_name = true)
    UNION ALL BY NAME
    SELECT 'green' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           lpep_pickup_datetime AS pickup, lpep_dropoff_datetime AS dropoff, *
               EXCLUDE (lpep_pickup_datetime, lpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/green/2026/*.parquet', filename = true, union_by_name = true)
)
SELECT tipo,
       count(*) FILTER (WHERE passenger_count = 0) AS pasajeros_cero,
       count(*) FILTER (WHERE passenger_count > 6) AS pasajeros_mas_de_6,
       count(*) FILTER (WHERE RatecodeID NOT

,tipo,pasajeros_cero,pasajeros_mas_de_6,ratecode_fuera_dic,payment_fuera_dic,payment_cero,zona_desconocida,vendor_ids
0,green,4527,99,2,0,0,5886,"[1, 2, 6]"
1,yellow,91359,28,769693,0,7716688,201486,"[1, 2, 6, 7]"


In [16]:
ejecutar("15_duplicados")

-- 3.6 Registros exactamente duplicados (todas las columnas iguales)
WITH yellow AS (
    SELECT * FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true)
), green AS (
    SELECT * FROM read_parquet('data/raw/green/2026/*.parquet', union_by_name = true)
)
SELECT 'yellow' AS tipo, count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM yellow)) AS filas_duplicadas
FROM yellow
UNION ALL
SELECT 'green', count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM green))
FROM green;



,tipo,filas_duplicadas
0,yellow,7
1,green,0


In [17]:
ejecutar("16_patron_nulos")

-- 3.6 Perfil de los registros con passenger_count nulo: ¿se concentran en algun proveedor o forma de pago?
WITH viajes AS (
    SELECT 'yellow' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           tpep_pickup_datetime AS pickup, tpep_dropoff_datetime AS dropoff, *
               EXCLUDE (tpep_pickup_datetime, tpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/yellow/2026/*.parquet', filename = true, union_by_name = true)
    UNION ALL BY NAME
    SELECT 'green' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           lpep_pickup_datetime AS pickup, lpep_dropoff_datetime AS dropoff, *
               EXCLUDE (lpep_pickup_datetime, lpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/green/2026/*.parquet', filename = true, union_by_name = true)
)
SELECT tipo, VendorID, payment_type, RatecodeID,
       count(*) AS registros,
       count(*) FILTER (WHERE store_and_fwd_flag IS NULL AND congestio

,tipo,VendorID,payment_type,RatecodeID,registros,tambien_nulos
0,yellow,2,0,<NA>,6761917,6761917
1,yellow,1,0,<NA>,895381,895381
2,yellow,6,0,<NA>,59390,59390
3,green,6,<NA>,<NA>,34847,34847
4,green,2,<NA>,<NA>,13400,13400
5,green,1,<NA>,<NA>,528,528


In [18]:
ejecutar("17_ratecode_payment")

-- 3.6 Distribucion de RatecodeID y payment_type para entender los codigos fuera del diccionario
WITH viajes AS (
    SELECT 'yellow' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           tpep_pickup_datetime AS pickup, tpep_dropoff_datetime AS dropoff, *
               EXCLUDE (tpep_pickup_datetime, tpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/yellow/2026/*.parquet', filename = true, union_by_name = true)
    UNION ALL BY NAME
    SELECT 'green' AS tipo, regexp_extract(filename, '(\d{4}-\d{2})\.parquet', 1) AS mes_archivo,
           lpep_pickup_datetime AS pickup, lpep_dropoff_datetime AS dropoff, *
               EXCLUDE (lpep_pickup_datetime, lpep_dropoff_datetime, filename)
    FROM read_parquet('data/raw/green/2026/*.parquet', filename = true, union_by_name = true)
)
SELECT tipo, 'RatecodeID' AS columna, RatecodeID AS valor, count(*) AS registros
FROM viajes GROUP BY ALL
UNION ALL
SELECT tipo, 'payment_type', payment_type, c

,tipo,columna,valor,registros
0,green,RatecodeID,1,269152
1,green,RatecodeID,2,887
2,green,RatecodeID,3,203
3,green,RatecodeID,4,354
4,green,RatecodeID,5,17739
5,green,RatecodeID,6,2
6,green,RatecodeID,99,2
7,green,RatecodeID,<NA>,48775
8,green,payment_type,1,219980
9,green,payment_type,2,65921
